## Assignment 3 Useful Concepts Walkthrough
This notebook covers the concepts that will be useful for solving the Assignment 2 problems:
1. Structured Outputs
2. Tavily
3. Tool Calling
4. Agent Calling Tools (ReAct Agent)
5. How to do RAG in LangGraph
6. Running Evals Experiment with Arize

In [ ]:
from dotenv import load_dotenv

load_dotenv()

import os
print(os.environ['TAVILY_API_KEY'][:20])
print(os.environ['OPENAI_API_KEY'][:20])


## Structured Outputs
Force the LLM to return the output in a specific format (Pydantic class) instead of just a single text. This is extremely useful for integrating LLMs into your applications in a seamless way.

Read more about that here - https://python.langchain.com/docs/concepts/structured_outputs/.

In [ ]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

model = init_chat_model("gpt-5-mini", model_provider="openai", reasoning_effort="minimal")

class ClassificationResult(BaseModel):
    category: str = Field(description="The category of the user question")
    explanation: str = Field(description="The explanation of why the category is chosen")

model = model.with_structured_output(ClassificationResult)
prompt = ChatPromptTemplate.from_template("""
Take the user question and classify it into one of the following categories:
Geography: Questions about countries, cities, or geographical features.
Science: Questions about science, technology, or scientific concepts.
Math: Questions about math, algebra, geometry, or mathematical concepts.

User question: {question}
""")

chain = prompt | model
result = chain.invoke({"question": "What is the capital of France?"})
print(type(result))
print(result)

In [ ]:
# Without structured output, LLM will return only the text.
plain_model = init_chat_model("gpt-5-mini", model_provider="openai", reasoning_effort="minimal")
chain = prompt | plain_model
model_only_result = chain.invoke({"question": "What is the capital of France?"})
print(type(model_only_result))
print(model_only_result)
print(model_only_result.content)

Structured output can also work with more complex data structures like lists as well. See the below example for identifying all places user asks in a question.

In [ ]:
from pydantic import BaseModel, Field

class Places(BaseModel):
    places: list[str] = Field(description="List of places")

model = init_chat_model("gpt-5-mini", model_provider="openai", reasoning_effort="minimal")

model = model.with_structured_output(Places)

# NOTE: This example is not answering the question, just extracting information
# from the question. You can also use it to answer the question.
prompt = ChatPromptTemplate.from_template("""
List all the places that the user asked in the question: {question}
""")

chain = prompt | model
result = chain.invoke({"question": "What is the capital of France and India?"})
print(type(result))
print(result)

## Tavily Web Search Example

In [ ]:
from langchain_tavily import TavilySearch

tool = TavilySearch(
    max_results=3,
    include_answer=True,  # True provides an answer from Tavily's side. Use False in assignments.
    include_raw_content=True,
    include_images=False,
    search_depth="advanced",  # advanced/basic are the two options
)

In [ ]:
# Calling the tool directly
tool.invoke({"query": "What are some latest innovations in AI?"})

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph
from IPython.display import Image
from langgraph.graph import START, END

class WebSearchState(TypedDict):
    query: str
    results: dict
    char_counts: list[int]

def web_search(state: WebSearchState):
    results = tool.invoke({"query": state["query"]})
    return {"results": results}

# Counts the number of characters in the raw content of each result:
def sum_char_counts(state: WebSearchState):
    char_counts = [len(result['raw_content']) for result in state["results"]["results"]]
    return {"char_counts": char_counts}

graph_builder = StateGraph(WebSearchState)

graph_builder.add_node("web_search", web_search)
graph_builder.add_node("sum_char_counts", sum_char_counts)

graph_builder.add_edge(START, "web_search")
graph_builder.add_edge("web_search", "sum_char_counts")
graph_builder.add_edge("sum_char_counts", END)

graph = graph_builder.compile()

In [ ]:
# Display the graph
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
final_result = graph.invoke({"query": "What are some latest innovations in AI?"})
print(final_result['char_counts'])

## Tool Calling

In the Tavily example above, we called the search tool **directly from our code**. But a key LLM capability is **tool calling** — where the LLM itself decides *when* and *how* to call a tool.

The flow works like this:
1. You define tools (Python functions with descriptions)
2. You bind them to the LLM with `.bind_tools()`
3. The LLM receives a user question and **decides** whether to call a tool
4. If it does, it returns a structured `tool_calls` request (not the actual result!)
5. Your code executes the tool and feeds the result back as a `ToolMessage`
6. The LLM uses the tool result to generate a final answer

**Key insight:** The LLM does NOT execute tools — it *requests* tool calls. Your code is responsible for executing them.

Read more: https://python.langchain.com/docs/concepts/tool_calling/

#### Step 1: Define tools using the `@tool` decorator

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

# Each tool has a name, description, and input schema (auto-generated from the function signature)
print(f"Tool name: {multiply.name}")
print(f"Tool description: {multiply.description}")
print(f"Tool input schema: {multiply.args_schema.model_json_schema()}")

#### Step 2: Bind tools to the LLM

This tells the LLM what tools are available. The LLM will decide on its own whether to use them.

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-5-mini", model_provider="openai", reasoning_effort="minimal")

# Bind tools to the model — now the LLM knows about multiply and add
llm_with_tools = llm.bind_tools([multiply, add])

#### Step 3: Invoke the LLM — it decides to call a tool

When you ask a question that requires computation, the LLM will **not** try to answer directly. Instead, it returns a `tool_calls` request telling you which tool to run and with what arguments.

In [ ]:
from langchain_core.messages import HumanMessage

# Ask a question that requires the multiply tool
result = llm_with_tools.invoke([HumanMessage(content="What is 1847 * 293 + 5812?")])

# The LLM does NOT return answer — it returns a tool call request!
print("Content (may be empty):", repr(result.content))
print("\nTool calls:")
for tc in result.tool_calls:
    print(f"  Tool: {tc['name']}")
    print(f"  Args: {tc['args']}")
    print(f"  ID:   {tc['id']}")

#### Step 4: Execute the tool and feed the result back

Now we execute the tool ourselves and create a `ToolMessage` with the result. Then we pass the full conversation (human → AI tool request → tool result) back to the LLM so it can generate a final answer.

In [ ]:
from langchain_core.messages import ToolMessage

# Get the tool call details from the AI response
tool_call = result.tool_calls[0]

# Execute the tool (in this case, multiply)
tool_result = multiply.invoke(tool_call["args"])
print(f"Tool result: {tool_result}")

# Create a ToolMessage with the result — the tool_call_id links it to the request
tool_message = ToolMessage(
    content=str(tool_result),
    tool_call_id=tool_call["id"]
)

# Pass the full conversation back to the LLM:
# Human question → AI tool call request → Tool result
final_response = llm_with_tools.invoke([
    HumanMessage(content="What is 1847 * 293 + 5812?"),
    result,          # The AI's tool call request
    tool_message     # The tool's result
])

print(f"\nFinal Response: {final_response}")

##### When the LLM does NOT call a tool
If the question doesn't need any tools, the LLM will just answer directly — `tool_calls` will be empty.

In [ ]:
# A question that doesn't need tools
result_no_tool = llm_with_tools.invoke([HumanMessage(content="What is the capital of France?")])

print(f"Content: {result_no_tool.content}")
print(f"Tool calls: {result_no_tool.tool_calls}")  # Empty list — no tools needed

#### Using Tavily as a bound tool

Now let's combine what we learned — bind the **Tavily search** tool to the LLM so the LLM can decide when to search the web.

In [ ]:
from langchain_tavily import TavilySearch

@tool
def tavily_tool(query: str) -> str:
    """Call this tool for any questions related to 2026 events"""
    search_result = TavilySearch(max_results=3, search_depth="advanced", include_answer=True).invoke({"query": query})
    return search_result['answer']

# print(tavily_tool.invoke({"query": "Who won the world series in 2026?"}))

# Bind Tavily to the LLM
llm_with_search = llm.bind_tools([tavily_tool])

# The LLM decides whether it needs to search
result = llm_with_search.invoke([HumanMessage(content="Who won the world series in 2026?")])
print(result)
print(result.tool_calls)

## Agent Calling Tools (ReAct Agent)

In the Tool Calling section above, we manually handled the loop:
1. LLM requests a tool call
2. We execute the tool
3. We feed the result back

An **agent** automates this entire loop. Langchain's `create_agent` builds a **ReAct** (Reason + Act) agent that:
- Receives a question
- Decides which tool to call (if any)
- Executes the tool automatically
- Observes the result
- Decides whether to call another tool or return a final answer
- Repeats until done

This is the pattern used in Part 3 (Agentic RAG) of the assignment.

Read more: https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/

### Step 1: Create a ReAct agent with tools

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

# Define our tools
@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression. Example: '2 + 3 * 4'"""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

tavily_search = TavilySearch(max_results=3, search_depth="basic")
@tool
def tavily_tool(query: str) -> str:
    """Call this tool for any questions related to 2026 events"""
    search_result = TavilySearch(max_results=3, search_depth="advanced", include_answer=True).invoke({"query": query})
    return search_result['answer']

# Create the agent — it automatically handles the tool-calling loop
agent_llm = init_chat_model("gpt-5-mini", model_provider="openai", reasoning_effort="minimal")

agent = create_agent(
    agent_llm,
    tools=[calculator, tavily_search],
    system_prompt="You are a helpful assistant. Use your tools to answer questions accurately. ALWAYS use the tavily_search tool for any factual or real-world questions — never answer from memory."
)

print("Agent created with tools:", [t.name for t in [calculator, tavily_tool]])

### Step 2: Visualize the agent graph

The ReAct agent is itself a LangGraph with a built-in loop: `agent` → `tools` → `agent` → ... → `END`

In [ ]:
from IPython.display import Image

display(Image(agent.get_graph().draw_mermaid_png()))

### Step 3: Invoke the agent — it handles everything automatically

Unlike manual tool calling, you just pass a message and the agent handles the entire reason → act → observe loop.

In [ ]:
from langchain_core.messages import HumanMessage

# Agent uses the calculator tool automatically
result = agent.invoke({"messages": [HumanMessage(content="What is 15 * 23 + 47?")]})

# The final message contains the answer
print("Final answer:", result["messages"][-1].content)

### Step 4: Inspect the agent's message trace

The agent returns the full conversation including every tool call and result. This lets you see exactly how it reasoned.

In [ ]:
# Print the full message trace to see the agent's reasoning
for i, msg in enumerate(result["messages"]):
    msg_type = type(msg).__name__
    print(f"\n--- Message {i} ({msg_type}) ---")
    
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"Tool call: {tc['name']}({tc['args']})")
    
    if hasattr(msg, 'content') and msg.content:
        content_preview = msg.content[:200] + "..." if len(msg.content) > 200 else msg.content
        print(f"{content_preview}")

### Step 5: Agent with web search — multiple tool calls

The agent can also decide to call the search tool, and may even call multiple tools in sequence to build up a complete answer.

In [ ]:
# Agent decides to use web search for a factual question
result = agent.invoke({"messages": [HumanMessage(content="What is the current population of Tokyo?")]})

print("Final answer:", result["messages"][-1].content)
print(f"\nTotal messages in trace: {len(result['messages'])}")

# Show which tools were called
tools_used = [
    tc['name']
    for msg in result['messages']
    if hasattr(msg, 'tool_calls')
    for tc in msg.tool_calls
]
print(f"Tools used: {tools_used}")

<!-- ### Key Difference: Manual Tool Calling vs Agent

| | Manual Tool Calling | Agent (ReAct) |
|---|---|---|
| **Who decides to call tools?** | LLM (via `bind_tools`) | LLM (via `create_react_agent`) |
| **Who executes tools?** | Your code (manually) | Agent (automatically) |
| **Who loops?** | Your code (manually) | Agent (automatically) |
| **Multi-step reasoning?** | You must code it | Built-in |
| **Use case** | Simple, single tool calls | Complex, multi-step tasks | -->

## RAG Example in LangGraph

In [ ]:
# Example of loading a PDF file:
# Refer here for more examples of document loaders: https://python.langchain.com/docs/how_to/document_loader_pdf/
from langchain_community.document_loaders import PyPDFLoader

import os
file_path = os.path.join(os.getcwd(), "code", "perplexia_ai", "docs", "2019-annual-performance-report.pdf")
loader = PyPDFLoader(file_path)
pages = []
async for page in loader.alazy_load():
    pages.append(page)

In [ ]:
print(len(pages))
test_page = pages[10]
print(type(test_page))
print(f"content: {test_page.page_content[:100]}")
print(f"metadata: {test_page.metadata}")
print(f"source: {test_page.metadata['source']}")


## Indexing the PDF file into a vector store

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma(
            embedding_function=embeddings,
            persist_directory="./chroma_db",
            collection_name="opm_documents_1"
        )

# Add the chunks (pages) from the PDF to the vector store
vector_store.add_documents(pages)

# Now perform similarity search
docs = vector_store.similarity_search("What are OPM's goals?", k=4)
print(len(docs))

In [ ]:
docs = vector_store.similarity_search("Who is the director of OPM?", k=4)

for doc in docs:
    print(doc.page_content)
    print("\n")


## Retrieval in RAG

In [ ]:
from typing import TypedDict

# Create a retriever object on the vector store
retriever = vector_store.as_retriever()

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate

model = init_chat_model("gpt-5-mini", model_provider="openai", reasoning_effort="minimal")

class GraphState(TypedDict):
    question: str
    documents: list[str]
    generated_answer: str

def retrieve_documents(state: GraphState):
    docs = vector_store.similarity_search(state["question"], k=4)
    return {"documents": docs}

def generate_answer(state: GraphState):
    # Call the model to generate the answer using the retrieved documents
    # and the question
    # Join all the retrieved documents into a single string
    prompt = PromptTemplate.from_template(
        "Answer the question based on the context provided. \n"
        "Question: {question}\n"
        "Context: {context}\n"
        "Answer: "
    )
    context = "\n".join([doc.page_content for doc in state["documents"]])
    prompt = prompt.format(question=state["question"], context=context)
    answer = model.invoke(prompt).content
    return {"generated_answer": answer}

graph_builder = StateGraph(GraphState)

graph_builder.add_node("retrieve_documents", retrieve_documents)
graph_builder.add_node("generate_answer", generate_answer)

graph_builder.add_edge(START, "retrieve_documents")
graph_builder.add_edge("retrieve_documents", "generate_answer")
graph_builder.add_edge("generate_answer", END)

graph = graph_builder.compile()

In [ ]:
# Display the graph
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
result = graph.invoke({"question": "What are OPM's goals?"})
print(result['generated_answer'])

In [ ]:
print(result['documents'])

In [ ]:
graph.invoke({"question": "Who won the baseball game today?"})

## Arize Experiment: Two-Step Joke Pipeline
A LangGraph pipeline that drafts a joke and then improves it, evaluated via Arize experiments.

In [ ]:
## State Model

from pydantic import BaseModel, Field, ConfigDict

class JokeGraphState(BaseModel):
    user_input: str
    intermediate_output: str = ""
    final_output: str = ""


In [ ]:
## Build the Graph

from langchain.chat_models import init_chat_model
from langgraph.graph import END, START, StateGraph

draft_llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0.9)
editor_llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0.7)

def first_draft(state):
    r = draft_llm.invoke(f"You are a comedy assistant.\nTopic: {state.user_input}\nWrite one short joke.")
    return {"intermediate_output": r.content.strip()}

def improve(state):
    r = editor_llm.invoke(f"You are a comedy editor.\nTopic: {state.user_input}\nDraft: {state.intermediate_output}\nRewrite into a stronger joke.")
    return {"final_output": r.content.strip()}

g = StateGraph(JokeGraphState)
g.add_node("draft", first_draft)
g.add_node("improve", improve)
g.add_edge(START, "draft")
g.add_edge("draft", "improve")
g.add_edge("improve", END)
joke_pipeline = g.compile()


In [ ]:
## Quick test

test = joke_pipeline.invoke({"user_input": "programmers"})
print(f"Draft: {test['intermediate_output']}\n\nFinal: {test['final_output']}")

# Sanity check — run the pipeline on a single topic to verify both steps produce output before connecting to Arize.


In [ ]:
## Evaluator

from arize.experiments import EvaluationResult
from phoenix.evals import LLM, create_classifier

eval_llm = LLM(provider="openai", model="gpt-4o-mini")
classifier = create_classifier(
    name="joke_quality", llm=eval_llm,
    prompt_template="Topic: {user_input}\nDraft: {intermediate_output}\nFinal: {final_output}\n\nIs the final joke on-topic AND improved? Return one label.",
    choices={"improved_and_on_topic": 1, "not_improved_or_off_topic": 0},
)

# `create_classifier` builds an LLM-as-judge evaluator. 
# It scores each joke pair on two criteria: was the final joke **on-topic** 
# and was it a **clear improvement** over the draft? Returns a binary label.



In [ ]:
# Patch the event loop so async evals run faster inside notebooks
# import nest_asyncio
# nest_asyncio.apply()

from arize import ArizeClient
from datetime import datetime, timezone

# Parse Arize dataset rows — maps the column "attributes.input.value" to user_input
class Row(BaseModel):
    model_config = ConfigDict(populate_by_name=True, extra="ignore")
    user_input: str = Field(alias="attributes.input.value")

# Task: run the joke pipeline on each dataset row
def task(dataset_row):
    inp = Row.model_validate(dataset_row).user_input
    r = joke_pipeline.invoke({"user_input": inp})
    return {"user_input": inp, "intermediate_output": r["intermediate_output"], "final_output": r["final_output"]}

# Evaluator: judge each pipeline output using the classifier from previous cell
def evaluator(dataset_row, output):
    s = classifier.evaluate({"user_input": Row.model_validate(dataset_row).user_input, "intermediate_output": output["intermediate_output"], "final_output": output["final_output"]})[0]
    return EvaluationResult(score=float(s.score) if s.score else None, label=s.label, explanation=s.explanation)

# Connect to Arize and find the dataset
client = ArizeClient(api_key=os.environ["ARIZE_API_KEY"])
ds = next(d for d in client.datasets.list(limit=100).datasets if d.name == "Joke Generator")

# Run the experiment — processes all rows, evaluates, and uploads results
exp, df = client.experiments.run(name=f"joke-{datetime.now(timezone.utc).strftime('%H%M%S')}", dataset_id=ds.id, task=task, evaluators={"judge": evaluator}, concurrency=1, timeout=120)
print(f"✅ {exp.id} — {len(df)} rows")
